<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/9wynik_koncowy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_wynik_koncowy.py

from __future__ import annotations

import json
from dataclasses import dataclass, field
from datetime import datetime
from enum import Flag, StrEnum, auto, verify, UNIQUE
from pathlib import Path
from typing import Any, Final


FOLDER_PROJEKTU: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)


@verify(UNIQUE)
class StatusWyniku(StrEnum):
    PASSED = "passed"
    FAILED = "failed"


class WarunkiWyniku(Flag):
    BRAK = 0

    PRZESZEDL_SKANOWANIE = auto()
    MA_PUNKTACJE = auto()
    MA_POZYCJE_W_RANKINGU = auto()
    WSZYSTKIE_FILTRY_PASSED = auto()


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True
)
class WynikFiltraKoncowy:
    typ: str
    status: StatusWyniku
    wartosc: float
    prog: float
    opis: str = ""


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True
)
class WynikSkanowania:
    ticker: str

    status: StatusWyniku

    wyniki_filtrow: list[
        WynikFiltraKoncowy
    ] = field(
        default_factory=list
    )

    wynik_punktowy: float = 0.0

    pozycja_w_rankingu: int | None = None

    poziom_jakosci: str | None = None

    poziom_jakosci_wartosc: int | None = None

    warunki: WarunkiWyniku = (
        WarunkiWyniku.BRAK
    )

    timestamp: datetime = field(
        default_factory=datetime.now,
        compare=False,
        repr=False
    )

    def __repr__(self) -> str:

        return (
            "WynikSkanowania("
            f"ticker='{self.ticker}', "
            f"status={self.status.name}, "
            f"wynik_punktowy={self.wynik_punktowy}, "
            f"pozycja_w_rankingu={self.pozycja_w_rankingu}, "
            f"poziom_jakosci={self.poziom_jakosci}, "
            f"liczba_filtrow={len(self.wyniki_filtrow)}"
            ")"
        )


def wczytaj_skaner(
    ticker: str,
    folder: Path = FOLDER_PROJEKTU
) -> dict[str, Any]:

    ticker = ticker.strip().upper()

    plik: Path = (
        folder
        / f"{ticker}_skaner.json"
    )

    if not plik.exists():
        raise FileNotFoundError(
            f"brak pliku z modulu 7: {plik}"
        )

    with open(
        plik,
        "r",
        encoding="utf-8"
    ) as f:

        dane: Any = json.load(f)

    if not isinstance(
        dane,
        dict
    ):
        raise ValueError(
            "dane skanera musza byc dict"
        )

    return dane


def wczytaj_ranking(
    ticker: str,
    folder: Path = FOLDER_PROJEKTU
) -> dict[str, Any]:

    ticker = ticker.strip().upper()

    plik: Path = (
        folder
        / f"{ticker}_ranking.json"
    )

    if not plik.exists():
        raise FileNotFoundError(
            f"brak pliku z modulu 8: {plik}"
        )

    with open(
        plik,
        "r",
        encoding="utf-8"
    ) as f:

        dane: Any = json.load(f)

    if not isinstance(
        dane,
        dict
    ):
        raise ValueError(
            "dane rankingu musza byc dict"
        )

    return dane


def parse_status(
    wartosc: str
) -> StatusWyniku:

    try:
        return StatusWyniku(
            wartosc
        )

    except ValueError as e:
        raise ValueError(
            f"nieznany status wyniku: {wartosc}"
        ) from e


def zbuduj_wyniki_filtrow(
    dane_skanera: dict[str, Any]
) -> list[WynikFiltraKoncowy]:

    rekordy: Any = (
        dane_skanera.get(
            "wyniki_filtrow",
            []
        )
    )

    if not isinstance(
        rekordy,
        list
    ):
        raise ValueError(
            "wyniki_filtrow musza byc lista"
        )

    wyniki: list[
        WynikFiltraKoncowy
    ] = []

    for rekord in rekordy:

        if not isinstance(
            rekord,
            dict
        ):
            raise ValueError(
                "wynik filtra musi byc dict"
            )

        wynik = WynikFiltraKoncowy(
            typ=str(
                rekord["typ"]
            ),

            status=parse_status(
                str(
                    rekord["status"]
                )
            ),

            wartosc=float(
                rekord["wartosc"]
            ),

            prog=float(
                rekord["prog"]
            ),

            opis=str(
                rekord.get(
                    "opis",
                    ""
                )
            )
        )

        wyniki.append(
            wynik
        )

    return wyniki


def okresl_warunki(
    status: StatusWyniku,
    wynik_punktowy: float,
    pozycja: int | None,
    wyniki_filtrow: list[
        WynikFiltraKoncowy
    ]
) -> WarunkiWyniku:

    warunki = (
        WarunkiWyniku.BRAK
    )

    if status == StatusWyniku.PASSED:

        warunki |= (
            WarunkiWyniku
            .PRZESZEDL_SKANOWANIE
        )

    if wynik_punktowy > 0:

        warunki |= (
            WarunkiWyniku
            .MA_PUNKTACJE
        )

    if pozycja is not None:

        warunki |= (
            WarunkiWyniku
            .MA_POZYCJE_W_RANKINGU
        )

    if (
        wyniki_filtrow
        and all(
            wynik.status
            == StatusWyniku.PASSED
            for wynik
            in wyniki_filtrow
        )
    ):

        warunki |= (
            WarunkiWyniku
            .WSZYSTKIE_FILTRY_PASSED
        )

    return warunki


def zbuduj_wynik_koncowy(
    ticker: str,
    dane_skanera: dict[str, Any],
    dane_rankingu: dict[str, Any]
) -> WynikSkanowania:

    ticker = ticker.strip().upper()

    przeszedl: bool = bool(
        dane_skanera[
            "przeszedl"
        ]
    )

    status: StatusWyniku = (
        StatusWyniku.PASSED
        if przeszedl
        else StatusWyniku.FAILED
    )

    wyniki_filtrow: list[
        WynikFiltraKoncowy
    ] = zbuduj_wyniki_filtrow(
        dane_skanera
    )

    wynik_punktowy: float = float(
        dane_rankingu[
            "wynik_punktowy"
        ]
    )

    pozycja: int | None = (
        int(
            dane_rankingu[
                "pozycja"
            ]
        )
        if dane_rankingu.get(
            "pozycja"
        ) is not None
        else None
    )

    poziom_raw: Any = (
        dane_rankingu.get(
            "poziom"
        )
    )

    poziom_nazwa: str | None = None
    poziom_wartosc: int | None = None

    if isinstance(
        poziom_raw,
        dict
    ):

        if (
            poziom_raw.get(
                "nazwa"
            )
            is not None
        ):
            poziom_nazwa = str(
                poziom_raw[
                    "nazwa"
                ]
            )

        if (
            poziom_raw.get(
                "wartosc"
            )
            is not None
        ):
            poziom_wartosc = int(
                poziom_raw[
                    "wartosc"
                ]
            )

    warunki: WarunkiWyniku = (
        okresl_warunki(
            status,
            wynik_punktowy,
            pozycja,
            wyniki_filtrow
        )
    )

    return WynikSkanowania(
        ticker=ticker,

        status=status,

        wyniki_filtrow=(
            wyniki_filtrow
        ),

        wynik_punktowy=(
            wynik_punktowy
        ),

        pozycja_w_rankingu=(
            pozycja
        ),

        poziom_jakosci=(
            poziom_nazwa
        ),

        poziom_jakosci_wartosc=(
            poziom_wartosc
        ),

        warunki=warunki
    )


def wynik_do_dict(
    wynik: WynikSkanowania
) -> dict[str, Any]:

    return {

        "ticker":
            wynik.ticker,

        "status":
            wynik.status.value,

        "wynik_punktowy":
            wynik.wynik_punktowy,

        "pozycja_w_rankingu":
            wynik.pozycja_w_rankingu,

        "poziom_jakosci": {
            "nazwa":
                wynik.poziom_jakosci,

            "wartosc":
                wynik.poziom_jakosci_wartosc
        },

        "warunki": {
            "wartosc":
                wynik.warunki.value,

            "spelnione": [
                warunek.name

                for warunek
                in WarunkiWyniku

                if (
                    warunek
                    != WarunkiWyniku.BRAK
                    and warunek
                    in wynik.warunki
                )
            ]
        },

        "wyniki_filtrow": [

            {
                "typ":
                    filtr.typ,

                "status":
                    filtr.status.value,

                "wartosc":
                    filtr.wartosc,

                "prog":
                    filtr.prog,

                "opis":
                    filtr.opis
            }

            for filtr
            in wynik.wyniki_filtrow
        ],

        "timestamp":
            wynik.timestamp.isoformat()
    }


def zapisz_wynik_koncowy(
    wynik: WynikSkanowania,
    folder: Path = FOLDER_PROJEKTU
) -> Path:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

    plik: Path = (
        folder
        / f"{wynik.ticker}_wynik_koncowy.json"
    )

    dane: dict[str, Any] = (
        wynik_do_dict(
            wynik
        )
    )

    with open(
        plik,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            dane,
            f,
            ensure_ascii=False,
            indent=2
        )

    return plik


def run() -> None:

    ticker: str = input(
        "podaj ticker: "
    ).strip().upper()

    print(
        "\nwczytywanie danych "
        "z modulu 7 i 8..."
    )

    dane_skanera: dict[
        str,
        Any
    ] = wczytaj_skaner(
        ticker
    )

    dane_rankingu: dict[
        str,
        Any
    ] = wczytaj_ranking(
        ticker
    )

    wynik: WynikSkanowania = (
        zbuduj_wynik_koncowy(
            ticker,
            dane_skanera,
            dane_rankingu
        )
    )

    print(
        "\nWYNIK KONCOWY"
    )

    print(
        wynik
    )

    print(
        "\nTICKER:",
        wynik.ticker
    )

    print(
        "STATUS:",
        wynik.status.value
    )

    print(
        "WYNIK PUNKTOWY:",
        wynik.wynik_punktowy
    )

    print(
        "POZYCJA W RANKINGU:",
        wynik.pozycja_w_rankingu
    )

    print(
        "POZIOM JAKOSCI:",
        wynik.poziom_jakosci
    )

    print(
        "\nWYNIKI FILTROW:"
    )

    for filtr in wynik.wyniki_filtrow:

        print(
            filtr.typ,
            "->",
            filtr.status.value,
            "| wartosc:",
            round(
                filtr.wartosc,
                4
            ),
            "| prog:",
            filtr.prog
        )

    print(
        "\nFLAGI WARUNKOW:"
    )

    for warunek in WarunkiWyniku:

        if (
            warunek
            != WarunkiWyniku.BRAK
            and warunek
            in wynik.warunki
        ):

            print(
                "-",
                warunek.name
            )

    plik: Path = (
        zapisz_wynik_koncowy(
            wynik
        )
    )

    print(
        "\nzapisano wynik koncowy:"
    )

    print(
        plik
    )

    print(
        "\nMODUL WYNIKU KONCOWEGO "
        "DZIALA POPRAWNIE"
    )